In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_filtered_set;
CREATE TABLE dev.mohit_gangwani.ad_labeling_filtered_set AS
SELECT vc.external_id AS ad_id
, NVL(vc.fk_dma_id, 0) AS fk_dma_id
, NULLIF(LOWER(TRIM(SPLIT_PART(REPLACE(brand_name, ',', '-'), '-', 1))), '[tbd]') AS brand
, NULLIF(LOWER(brand_name), '[tbd]') AS brand_name
, COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) AS total_impressions
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.location l
  ON l.location_id = vc.fk_location_id
 AND l.country_code = 'US'
JOIN (
  SELECT cief.fk_commercial_id, cief.external_id, cief.brand_name
  FROM prod.detection.commercial_id_external_firehose cief
  JOIN prod.detection.clients cl
    ON cl.client_id = cief.fk_client_id
  WHERE cl.client_name = 'kinetiq'
  GROUP BY ALL
) cief
  ON cief.external_id = vc.external_id
WHERE vc.session_start >= CURRENT_DATE - 8
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  -- AND (cief.external_id IS NOT NULL OR vc.fk_commercial_source_id = 2)
GROUP BY 1, 2, 3, 4
HAVING COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) >= 20
;

In [0]:
%sql
SELECT brand, brand_name, COUNT(*) FROM dev.mohit_gangwani.ad_labeling_filtered_set
-- WHERE brand LIKE '%toyota%'
GROUP BY 1, 2
ORDER BY 3 DESC
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_base_viewing_table_addressable;
CREATE TABLE dev.mohit_gangwani.ad_labeling_base_viewing_table_addressable AS
SELECT fk_tvid, session_start, session_end, ad_id, fk_dma_id, ds, brand, bin, is_new_pod
, SUM(is_new_pod) OVER (PARTITION BY fk_tvid ORDER BY session_start ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS pod_num
FROM (
  SELECT fk_tvid, session_start, session_end, ad_id, fk_dma_id, ds, brand, bin
  , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
  , CASE WHEN TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 120 OR prev_end IS NULL THEN 0 ELSE 1 END AS is_new_pod 
  FROM (
    SELECT vc.fk_tvid
    , vc.session_start
    , vc.session_end
    , COALESCE(vc.external_id, cief.ad_id) AS ad_id
    , COALESCE(cief.brand_name, vc.external_id, cief.ad_id) AS brand
    , NVL(vc.fk_dma_id, 0) AS fk_dma_id
    , DATE(vc.session_start) AS ds
    , TO_TIMESTAMP(FLOOR(UNIX_TIMESTAMP(vc.session_start) / 600) * 600) AS bin
    , ROW_NUMBER() OVER (PARTITION BY vc.fk_tvid, vc.session_start ORDER BY vc.session_start, vc.session_end DESC, vc.external_id DESC) AS rn
    FROM prod.detection.viewing_commercials_firehose_dedup vc
    JOIN dev.mohit_gangwani.ad_labeling_filtered_set cief
      ON cief.ad_id = vc.external_id
    AND cief.fk_dma_id = NVL(vc.fk_dma_id, 0)
    JOIN prod.detection.location l
      ON l.location_id = vc.fk_location_id
    AND l.country_code = 'US'
    LEFT JOIN prod.detection.viewing_content_firehose content
      ON vc.fk_tvid = content.fk_tvid
    AND vc.prev_session_start = content.session_start
    AND content.session_start >= CURRENT_DATE - 2
    WHERE vc.session_start >= CURRENT_DATE - 1
      AND vc.session_start < CURRENT_DATE
      AND vc.fk_zoo_id = 17
      AND (UPPER(COALESCE(vc.prev_content_type, content.content_type)) <=> 'STREAMING'
            OR COALESCE(vc.prev_vizio_epg_station, content.vizio_epg_station) IS NOT NULL
            OR UPPER(COALESCE(vc.reported_input_source, content.reported_input_source)) <=> 'APPS')
  )
WHERE rn = 1
);

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_addressable ORDER BY fk_tvid, session_start
LIMIT 1000

STEP 2: Coverage percentile per brand per 10-min bucket

Statistical rarity signal to replace hard thresholds

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_brand_bucket_coverage_percentiles_addressable;
CREATE TABLE dev.mohit_gangwani.ad_labeling_brand_bucket_coverage_percentiles_addressable AS
WITH brand_bucket AS (
  SELECT brand
  , bin
  , COUNT(DISTINCT fk_tvid) AS global_tvids_for_brand_in_bucket
  , COUNT(DISTINCT fk_tvid||'-'||session_start) AS global_plays_for_brand_in_bucket
  FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_addressable
  GROUP BY 1, 2
)
SELECT *,
PERCENT_RANK() OVER (PARTITION BY bin ORDER BY global_tvids_for_brand_in_bucket) AS coverage_percentile
FROM brand_bucket;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_brand_bucket_coverage_percentiles_addressable
ORDER BY brand
LIMIT 100

STEP 3: Pod-level composition metrics

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_pod_composition_addressable;
CREATE TABLE dev.mohit_gangwani.ad_labeling_pod_composition_addressable AS
WITH pod_stats AS (
  SELECT fk_tvid
  , ds
  , pod_num
  , MIN(session_start) AS pod_start
  , MAX(session_end) AS pod_end
  , COUNT(*) AS pod_ads
  , COUNT(DISTINCT brand) AS distinct_brands
  FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_addressable
  GROUP BY 1, 2, 3
)
, pod_brand_counts AS (
  SELECT fk_tvid
  , ds
  , pod_num
  , brand
  , COUNT(*) AS brand_ads
  FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_addressable
  GROUP BY 1, 2, 3, 4
)
, pod_top_share AS (
  SELECT fk_tvid
  , ds
  , pod_num
  , MAX(brand_ads) AS top_brand_ads
  FROM pod_brand_counts
  GROUP BY 1, 2, 3
)
SELECT s.fk_tvid
, s.ds
, s.pod_num
, s.pod_start
, s.pod_end
, s.pod_ads
, s.distinct_brands
, t.top_brand_ads
, (t.top_brand_ads * 1.0) / s.pod_ads AS top_brand_share
, TO_TIMESTAMP(FLOOR(UNIX_TIMESTAMP(s.pod_start) / (10 * 60)) * (10 * 60)) AS pod_bin
FROM pod_stats s
JOIN pod_top_share t
  ON s.fk_tvid = t.fk_tvid
 AND s.ds = t.ds
 AND s.pod_num = t.pod_num

In [0]:
%sql
SELECT pod_bin, COUNT(*) FROM dev.mohit_gangwani.ad_labeling_pod_composition_addressable
GROUP BY 1
ORDER BY 1
LIMIT 1000

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_pod_vod_like_addressable;
CREATE TABLE dev.mohit_gangwani.ad_labeling_pod_vod_like_addressable AS
WITH pod_brands AS (
  SELECT DISTINCT fk_tvid
  , ds
  , pod_num
  , brand
  FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_addressable
  GROUP BY ALL
)
-- Join brand coverage percentile for the pod's bucket
, pod_brand_cov AS (
  SELECT b.fk_tvid
  , b.ds
  , b.pod_num
  , b.brand
  , c.coverage_percentile
  FROM pod_brands b
  JOIN dev.mohit_gangwani.ad_labeling_pod_composition_addressable pc
    ON pc.fk_tvid = b.fk_tvid
   AND pc.ds = b.ds
   AND pc.pod_num = b.pod_num
  LEFT JOIN dev.mohit_gangwani.ad_labeling_brand_bucket_coverage_percentiles_addressable c
    ON b.brand = c.brand
   AND c.bin = pc.pod_bin
  GROUP BY ALL
)
-- Collapse to pod-level rarity score:
-- take MIN percentile across brands in pod (conservative)
, pod_rarity AS (
  SELECT fk_tvid
  , ds
  , pod_num
  , MIN(coverage_percentile) AS min_brand_coverage_percentile
  FROM pod_brand_cov
  GROUP BY 1, 2, 3
)
SELECT pc.*
, pr.min_brand_coverage_percentile
-- A) Multi-ad diverse pod (VOD-like)
-- B) Single-ad pod: brand is rare in this time bucket (bottom 5%)
, CASE WHEN pc.pod_ads >= 2 AND pc.distinct_brands >= 2 AND pc.top_brand_share <= 0.70 THEN 1
       WHEN pc.pod_ads = 1 AND pr.min_brand_coverage_percentile IS NOT NULL AND pr.min_brand_coverage_percentile <= 0.05 THEN 1
       ELSE 0
  END AS is_vod_like_pod
, CASE WHEN pc.pod_ads >= 2 AND pc.distinct_brands >= 2 AND pc.top_brand_share <= 0.70 THEN 'VOD_POD_DIVERSITY'
       WHEN pc.pod_ads = 1 AND pr.min_brand_coverage_percentile IS NOT NULL AND pr.min_brand_coverage_percentile <= 0.05 THEN 'VOD_SINGLE_AD_RARE_PCTL'
       ELSE NULL
  END AS vod_pod_reason
FROM dev.mohit_gangwani.ad_labeling_pod_composition_addressable pc
LEFT JOIN pod_rarity pr
  ON pc.fk_tvid = pr.fk_tvid
 AND pc.ds = pr.ds
 AND pc.pod_num = pr.pod_num;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_pod_vod_like_addressable
ORDER BY fk_tvid, pod_start
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_full_pod_labels_addressable;
CREATE TABLE dev.mohit_gangwani.ad_labeling_full_pod_labels_addressable AS
SELECT p.fk_tvid
, p.fk_dma_id
, p.ds
, p.session_start
, p.session_end
, p.ad_id
, p.brand
, p.pod_num
, v.is_vod_like_pod AS is_addressable_full_pod
, v.vod_pod_reason
, v.pod_ads
, v.distinct_brands
, v.top_brand_share
, v.min_brand_coverage_percentile
FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_addressable p
LEFT JOIN dev.mohit_gangwani.ad_labeling_pod_vod_like_addressable v
  ON p.fk_tvid = v.fk_tvid
 AND p.ds = v.ds
 AND p.pod_num = v.pod_num;

In [0]:
%sql
-- 1) Overall counts by reason
SELECT is_addressable_full_pod, vod_pod_reason, COUNT(*) AS n_plays
FROM dev.mohit_gangwani.ad_labeling_full_pod_labels_addressable
GROUP BY 1,2
ORDER BY 1 DESC, 2;

In [0]:
%sql
SELECT *
FROM dev.mohit_gangwani.ad_labeling_full_pod_labels_addressable
-- WHERE is_addressable_full_pod = 1
ORDER BY fk_tvid, session_start
LIMIT 1000;

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_final_table_addressable
WHERE fk_tvid = 20086944

In [0]:
sample_df = spark.table("dev.mohit_gangwani.ad_labeling_final_table_addressable_sample")
display(sample_df.orderBy('fk_tvid'))

In [0]:
summary = (
    sample_df.groupBy("sample_group", "is_addressable_session", "reason_code")
    .agg(
        F.count("*").alias("n_sessions"),
        F.expr("percentile_approx(plays_in_bin, 0.5)").alias("p50_plays_in_bin"),
        F.expr("percentile_approx(min_gap_seconds_in_bin, 0.5)").alias("p50_min_gap_s"),
        F.expr("percentile_approx(utilization_in_bin, 0.5)").alias("p50_utilization"),
        F.expr("percentile_approx(global_tvids_in_bucket, 0.5)").alias(
            "p50_global_breadth"
        ),
    )
    .orderBy("sample_group", "reason_code")
)

display(summary)

Addressables:
- Full ad pod wil 